In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q21/q21_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

# Select relevant columns from lineitem
li = lineitem[['L_ORDERKEY','L_SUPPKEY','L_RECEIPTDATE','L_COMMITDATE']]

# Pre-filter by date condition and keep minimal columns
date_li = li[li.L_RECEIPTDATE > li.L_COMMITDATE][['L_ORDERKEY','L_SUPPKEY']]

# Compute original and post-filter supplier counts per order
o_orig = li.groupby('L_ORDERKEY').L_SUPPKEY.nunique().reset_index().rename(columns={'L_SUPPKEY':'orig_count'})
o_new  = date_li.groupby('L_ORDERKEY').L_SUPPKEY.nunique().reset_index().rename(columns={'L_SUPPKEY':'new_count'})

# Identify orders with >1 original supplier and exactly one after the date filter
valid_orders = o_orig.merge(o_new, on='L_ORDERKEY', how='inner')
valid_orders = valid_orders[(valid_orders.orig_count > 1) & (valid_orders.new_count == 1)][['L_ORDERKEY']]

# Keep only those lineitems
filtered_li = date_li.merge(valid_orders, on='L_ORDERKEY', how='inner')

# Filter orders by status 'F'
o_f = orders[orders.O_ORDERSTATUS == 'F'][['O_ORDERKEY']]

# Join filtered lineitems with filtered orders
df = filtered_li.merge(o_f, left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')

# Build supplier→Saudi Arabia lookup
supp_nat = supplier[['S_SUPPKEY','S_NATIONKEY','S_NAME']].merge(
    nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
    left_on='S_NATIONKEY', right_on='N_NATIONKEY', how='inner'
)[['S_SUPPKEY','S_NAME']]

# Final join and aggregation
total = (
    df.merge(supp_nat, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')[['S_NAME']]
      .groupby('S_NAME', as_index=False)
      .size()
      .rename(columns={'size':'NUMWAIT'})
      .sort_values(by=['NUMWAIT','S_NAME'], ascending=[False,True])
)